In [ ]:
%cd ..

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from reinforcement_learning.rewards import reward_function_proposed


# Proposed (from PuRL)

In [ ]:
def plot_reward_surface(Tspars, Tdmap, spars_coeff, dmap_coeff, beta=5, device="cpu", resolution=100):
    # 1. Reward vs spars (fixed dmap)
    spars_vals = np.linspace(0, 1, resolution)
    spars_tensor = torch.tensor(spars_vals, dtype=torch.float32, device=device)
    dmap_fixed = torch.full_like(spars_tensor, Tdmap)
    reward_spars = reward_function_proposed(spars_tensor, Tspars, dmap_fixed, Tdmap, spars_coeff, dmap_coeff, beta).cpu().numpy()

    # 2. Reward vs dmap (fixed spars)
    dmap_vals = np.linspace(0, 1, resolution)
    dmap_tensor = torch.tensor(dmap_vals, dtype=torch.float32, device=device)
    spars_fixed = torch.full_like(dmap_tensor, Tspars)
    reward_dmap = reward_function_proposed(spars_fixed, Tspars, dmap_tensor, Tdmap, spars_coeff, dmap_coeff, beta).cpu().numpy()

    # 3. Reward vs (spars, dmap)
    spars_grid, dmap_grid = np.meshgrid(spars_vals, dmap_vals)
    spars_grid_tensor = torch.tensor(spars_grid, dtype=torch.float32, device=device)
    dmap_grid_tensor = torch.tensor(dmap_grid, dtype=torch.float32, device=device)
    reward_grid = reward_function_proposed(spars_grid_tensor, Tspars, dmap_grid_tensor, Tdmap, spars_coeff, dmap_coeff, beta).cpu().numpy()

    # Plotting
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))

    # Combined plot for Reward vs spars and Reward vs dmap
    axs[0].plot(spars_vals, reward_spars, label='spars (dmap = Tdmap)')
    axs[0].axvline(Tspars, color='gray', linestyle='--', label='Tspars')
    axs[0].plot(dmap_vals, reward_dmap, label='dmap (spars = Tspars)', color='orange')
    axs[0].axvline(Tdmap, color='red', linestyle='--', label='Tdmap')
    axs[0].set_xlabel("Value")
    axs[0].set_ylabel("Reward")
    axs[0].set_title("Reward vs spars & Reward vs dmap")
    axs[0].legend()

    # Reward surface plot
    cp = axs[1].contourf(spars_grid, dmap_grid, reward_grid, levels=50, cmap='plasma')
    fig.colorbar(cp, ax=axs[1], label="Reward")
    axs[1].set_xlabel("spars")
    axs[1].set_ylabel("dmap")
    axs[1].set_title("Reward Surface (dmap vs spars)")

    plt.tight_layout()
    plt.show()


In [ ]:
Tspars = 0.7
Tdmap = 0.05
spars_coeff = 0.5
dmap_coeff = 0.5
beta = 5

plot_reward_surface(Tspars, Tdmap, spars_coeff, dmap_coeff, beta)

# Partial Target Reward

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

def plot_partial_reward_surface(rewarder, layer_idx: int, device="cpu", resolution=100):
    """
    Plot reward vs spars (at dmap=Td), reward vs dmap (at spars=Ts[layer]),
    and the 2D reward surface for a given layer.
    """
    eps = 1e-12

    # Resolve scalar Td and Ts for visuals
    Td = rewarder.Tdmap
    if isinstance(Td, torch.Tensor):
        if Td.numel() != 1:
            raise ValueError("Tdmap must be a scalar for plotting.")
        Td = float(Td.detach().cpu().item())
    Ts = float(torch.clamp(rewarder.layer_Tspars[layer_idx], min=eps).detach().cpu().item())

    # Grids
    spars_vals = np.linspace(0, 1, resolution)
    dmap_vals  = np.linspace(0, 1, resolution)

    # Tensors
    spars_t = torch.tensor(spars_vals, dtype=torch.float32, device=device)
    dmap_t  = torch.tensor(dmap_vals,  dtype=torch.float32, device=device)

    # 1) Reward vs spars (fix dmap = Td)
    dmap_fixed = torch.full_like(spars_t, Td)
    r_vs_spars = rewarder.get_reward(layer_idx, spars_t, dmap_fixed).detach().cpu().numpy()

    # 2) Reward vs dmap (fix spars = Ts)
    spars_fixed = torch.full_like(dmap_t, Ts)
    r_vs_dmap = rewarder.get_reward(layer_idx, spars_fixed, dmap_t).detach().cpu().numpy()

    # 3) 2D surface (spars, dmap)
    S, D = np.meshgrid(spars_vals, dmap_vals)
    S_t = torch.tensor(S, dtype=torch.float32, device=device)
    D_t = torch.tensor(D, dtype=torch.float32, device=device)
    R = rewarder.get_reward(layer_idx, S_t, D_t).detach().cpu().numpy()

    # Plot
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))

    # Left: two 1D cuts
    axs[0].plot(spars_vals, r_vs_spars, label='Reward vs spars (dmap=Td)')
    axs[0].axvline(Ts, linestyle='--', color='gray', label='Ts (layer target)')
    axs[0].plot(dmap_vals, r_vs_dmap, label='Reward vs dmap (spars=Ts)')
    axs[0].axvline(Td, linestyle='--', color='red',  label='Td (target)')
    axs[0].set_xlabel("Value")
    axs[0].set_ylabel("Reward")
    axs[0].set_title(f"Layer {layer_idx}: Reward vs spars & dmap")
    axs[0].legend()

    # Right: surface
    cp = axs[1].contourf(S, D, R, levels=50)
    fig.colorbar(cp, ax=axs[1], label="Reward")
    axs[1].set_xlabel("spars")
    axs[1].set_ylabel("dmap")
    axs[1].set_title(f"Layer {layer_idx}: Reward Surface")

    plt.tight_layout()
    plt.show()


In [ ]:
from reinforcement_learning.rewards import  PartialTargetReward, SigmoidGateReward
from src.model.yolo_handler import YoloHandler
from utils.config_parser import ConfigParser

Tspars = 0.7
Tdmap = 0.05
spars_coeff = 0.2
dmap_coeff = 0.8
beta = 50

sample_conf =  ConfigParser.read("/home/blanka/Multi-Domain-Pruning/config/pruning/pruning_sampling.ini")
yolo_conf = sample_conf.model
yolo_handler = YoloHandler(yolo_conf)
rewarder = PartialTargetReward( yolo_handler.prunable_layers,
                                Tspars, Tdmap, beta, spars_coeff,dmap_coeff, "cuda")

plot_partial_reward_surface(rewarder, layer_idx=50, device="cpu", resolution=200)
